In [10]:
from supabase import create_client
from dotenv import load_dotenv
import os
import pandas as pd

load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# supabase päringu piirangud on 1000 rida, seega tuleb andmete saamiseks teha mitu päringut
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = (
            supabase.table(table_name)
            .select("*")
            .range(page * page_size, (page + 1) * page_size - 1)
            .execute()
        )

        data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(data)
# küsin andmeid puhastatud tabelitest
df_orders = get_data('sales')
df_customers = get_data('customers')
print(f"Tellimusi: {len(df_orders)}, Kliente: {len(df_customers)}")
# küsin andmeid puhastamata tabelitest
df_orders_raw = get_data('sales_py')
df_customers_raw = get_data('customers_py')
print(f"Tellimusi (puhastamata): {len(df_orders_raw)}, Kliente (puhastamata): {len(df_customers_raw)}")


Tellimusi: 10118, Kliente: 3150
Tellimusi (puhastamata): 15234, Kliente (puhastamata): 3150


In [13]:
response = supabase.table('sales').select('*').eq('store_location', 'Tallinn').order('total_price', desc=True).execute()
df = pd.DataFrame(response.data)
print(f"Tallinna tellimusi: {len(df)}, Käive: {df['total_price'].sum():.2f} EUR")

Tallinna tellimusi: 1000, Käive: 631891.67 EUR


In [14]:
linnad = ['Tallinn', 'Tartu', 'Pärnu']

for linn in linnad:
    response = (
        supabase.table('sales')
        .select('*')
        .eq('store_location', linn)
        .order('total_price', desc=True)
        .execute()
    )

    df = pd.DataFrame(response.data)

    print(f"{linn}: tellimusi {len(df)}, Käive: {df['total_price'].sum():.2f} EUR")

Tallinn: tellimusi 1000, Käive: 631891.67 EUR
Tartu: tellimusi 1000, Käive: 447055.39 EUR
Pärnu: tellimusi 1000, Käive: 292926.96 EUR
